In [ ]:
import urllib.request

# Don Quijote en español (~2MB) — perfecto para entrenar
urllib.request.urlretrieve(
    "https://www.gutenberg.org/files/2000/2000-0.txt",
    "input.txt"
)
print("input.txt descargado correctamente")

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from tqdm.notebook import tqdm 
import os

# ── Configuración ──────────────────────────────────────────────────────────────
CONFIG = {
    "tamaño_lote":      32,
    "longitud_bloque":  512,
    "max_iteraciones":  8000,
    "intervalo_eval":   400,
    "tasa_aprendizaje": 1e-4,
    "iteraciones_eval": 200,
    "n_embedding":      512,
    "n_cabezas":        8,
    "n_capas":          8,
    "abandono":         0.1,
    "temperatura":      0.8,
    "top_k":            40,
}

tamaño_lote      = CONFIG["tamaño_lote"]
longitud_bloque  = CONFIG["longitud_bloque"]
max_iteraciones  = CONFIG["max_iteraciones"]
intervalo_eval   = CONFIG["intervalo_eval"]
tasa_aprendizaje = CONFIG["tasa_aprendizaje"]
iteraciones_eval = CONFIG["iteraciones_eval"]
n_embedding      = CONFIG["n_embedding"]
n_cabezas        = CONFIG["n_cabezas"]
n_capas          = CONFIG["n_capas"]
abandono         = CONFIG["abandono"]
dispositivo      = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.manual_seed(1337)

# ── Dataset ────────────────────────────────────────────────────────────────────
with open('input.txt', 'r', encoding='utf-8') as f:
    texto = f.read()

caracteres         = sorted(list(set(texto)))
tamaño_vocabulario = len(caracteres)
caracter_a_numero  = {ch: i for i, ch in enumerate(caracteres)}
numero_a_caracter  = {i: ch for i, ch in enumerate(caracteres)}
codificar          = lambda s: [caracter_a_numero.get(c, 0) for c in s]
decodificar        = lambda l: ''.join([numero_a_caracter[i] for i in l])

datos               = torch.tensor(codificar(texto), dtype=torch.long)
n                   = int(0.9 * len(datos))
datos_entrenamiento = datos[:n]
datos_validacion    = datos[n:]

# ── Lote ───────────────────────────────────────────────────────────────────────
def obtener_lote(division):
    datos = datos_entrenamiento if division == 'train' else datos_validacion
    ix    = torch.randint(len(datos) - longitud_bloque, (tamaño_lote,))
    x     = torch.stack([datos[i:i + longitud_bloque] for i in ix])
    y     = torch.stack([datos[i + 1:i + longitud_bloque + 1] for i in ix])
    return x.to(dispositivo), y.to(dispositivo)

# ── Evaluación ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def estimar_perdida():
    resultado = {}
    modelo.eval()
    for division in ['train', 'val']:
        perdidas = torch.zeros(iteraciones_eval)
        for k in range(iteraciones_eval):
            X, Y = obtener_lote(division)
            _, perdida = modelo(X, Y)
            perdidas[k] = perdida.item()
        resultado[division] = perdidas.mean()
    modelo.train()
    return resultado

# ── Arquitectura ───────────────────────────────────────────────────────────────
class CabezaAtencion(nn.Module):
    def __init__(self, tamaño_cabeza):
        super().__init__()
        self.clave    = nn.Linear(n_embedding, tamaño_cabeza, bias=False)
        self.consulta = nn.Linear(n_embedding, tamaño_cabeza, bias=False)
        self.valor    = nn.Linear(n_embedding, tamaño_cabeza, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(longitud_bloque, longitud_bloque)))
        self.abandono = nn.Dropout(abandono)

    def forward(self, x):
        B, T, C = x.shape
        k = self.clave(x)
        q = self.consulta(x)
        pesos = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5
        pesos = pesos.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        pesos = F.softmax(pesos, dim=-1)
        pesos = self.abandono(pesos)
        return pesos @ self.valor(x)

class AtencionMultiCabeza(nn.Module):
    def __init__(self, num_cabezas, tamaño_cabeza):
        super().__init__()
        self.cabezas    = nn.ModuleList([CabezaAtencion(tamaño_cabeza) for _ in range(num_cabezas)])
        self.proyeccion = nn.Linear(tamaño_cabeza * num_cabezas, n_embedding)
        self.abandono   = nn.Dropout(abandono)

    def forward(self, x):
        return self.abandono(self.proyeccion(torch.cat([h(x) for h in self.cabezas], dim=-1)))

class RedAdelante(nn.Module):
    def __init__(self, n_embedding):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_embedding, 4 * n_embedding),
            nn.GELU(),
            nn.Linear(4 * n_embedding, n_embedding),
            nn.Dropout(abandono),
        )

    def forward(self, x):
        return self.red(x)

class BloqueTransformer(nn.Module):
    def __init__(self, n_embedding, n_cabezas):
        super().__init__()
        tamaño_cabeza      = n_embedding // n_cabezas
        self.auto_atencion = AtencionMultiCabeza(n_cabezas, tamaño_cabeza)
        self.red_adelante  = RedAdelante(n_embedding)
        self.norm1         = nn.LayerNorm(n_embedding)
        self.norm2         = nn.LayerNorm(n_embedding)

    def forward(self, x):
        x = x + self.auto_atencion(self.norm1(x))
        x = x + self.red_adelante(self.norm2(x))
        return x

class ModeloLenguajeGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb_tokens    = nn.Embedding(tamaño_vocabulario, n_embedding)
        self.emb_posiciones = nn.Embedding(longitud_bloque, n_embedding)
        self.bloques        = nn.Sequential(*[BloqueTransformer(n_embedding, n_cabezas) for _ in range(n_capas)])
        self.norm_final     = nn.LayerNorm(n_embedding)
        self.cabeza         = nn.Linear(n_embedding, tamaño_vocabulario)
        self.apply(self._init_pesos)

    def _init_pesos(self, modulo):
        if isinstance(modulo, nn.Linear):
            torch.nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
            if modulo.bias is not None:
                torch.nn.init.zeros_(modulo.bias)
        elif isinstance(modulo, nn.Embedding):
            torch.nn.init.normal_(modulo.weight, mean=0.0, std=0.02)

    def forward(self, idx, objetivos=None):
        B, T = idx.shape
        x = self.emb_tokens(idx) + self.emb_posiciones(torch.arange(T, device=dispositivo))
        x = self.norm_final(self.bloques(x))
        logits = self.cabeza(x)
        perdida = None
        if objetivos is not None:
            B, T, C = logits.shape
            perdida = F.cross_entropy(logits.view(B * T, C), objetivos.view(B * T))
        return logits, perdida

    @torch.no_grad()
    def generar(self, idx, max_tokens_nuevos, temperatura=0.8, top_k=40):
        for _ in range(max_tokens_nuevos):
            idx_cond = idx[:, -longitud_bloque:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperatura          #temperatura

            # ✅ Top-K sampling
            if top_k is not None:
                valores, _ = torch.topk(logits, top_k)
                logits[logits < valores[:, [-1]]] = float('-inf')

            probs     = F.softmax(logits, dim=-1)
            idx_next  = torch.multinomial(probs, num_samples=1)
            idx       = torch.cat((idx, idx_next), dim=1)
        return idx

# ── Inicializar modelo ─────────────────────────────────────────────────────────
modelo     = ModeloLenguajeGPT().to(dispositivo)
optimizador = torch.optim.AdamW(modelo.parameters(), lr=tasa_aprendizaje)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(   #LR scheduler
    optimizador, T_max=max_iteraciones
)

print(f"Modelo: {sum(p.numel() for p in modelo.parameters())/1e6:.1f}M parámetros")
print(f"Dispositivo: {dispositivo}")

# ── Cargar modelo existente ────────────────────────────────────────────────────
modelo_guardado = 'modelo_entrenado.pth'
entrenar_nuevo  = True

if os.path.exists(modelo_guardado):
    print(f"\nModelo encontrado: {modelo_guardado}")
    respuesta = input("¿Cargar modelo existente? (s/n): ").lower().strip()
    if respuesta in ('s', 'si', 'sí'):
        checkpoint = torch.load(modelo_guardado, map_location=dispositivo)
        modelo.load_state_dict(checkpoint['modelo_estado'])
        optimizador.load_state_dict(checkpoint['optimizador_estado'])
        print("Modelo cargado.")
        entrenar_nuevo = False
    else:
        print("Entrenando nuevo modelo...")

# ── Entrenamiento ──────────────────────────────────────────────────────────────
if entrenar_nuevo:
    print("\n🚀 Iniciando entrenamiento...\n")
    for iteracion in tqdm(range(max_iteraciones), desc="Entrenando"):   # tqdm

        if iteracion % intervalo_eval == 0 or iteracion == max_iteraciones - 1:
            perdidas = estimar_perdida()
            tqdm.write(f"  paso {iteracion:5d} | train: {perdidas['train']:.4f} | val: {perdidas['val']:.4f}")

        xb, yb = obtener_lote('train')
        logits, perdida = modelo(xb, yb)
        optimizador.zero_grad(set_to_none=True)
        perdida.backward()
        optimizador.step()
        scheduler.step()                                                 # scheduler

    # Guardar modelo
    torch.save({
        'modelo_estado':      modelo.state_dict(),
        'optimizador_estado': optimizador.state_dict(),
        'caracteres':         caracteres,
        'tamaño_vocabulario': tamaño_vocabulario,
        'longitud_bloque':    longitud_bloque,
        'n_embedding':        n_embedding,
        'n_cabezas':          n_cabezas,
        'n_capas':            n_capas,
        'abandono':           abandono,
    }, modelo_guardado)
    print(f"\n💾 Modelo guardado en '{modelo_guardado}'")

# ── Generación / Chat ──────────────────────────────────────────────────────────
modelo.eval()
print("\n Escribe un prompt para generar texto. ('salir' para terminar)\n")

while True:
    prompt = input("Tú: ").strip()
    if prompt.lower() in ('salir', 'exit', 'quit'):
        print("👋 Hasta luego.")
        break
    if not prompt:
        continue

    contexto = torch.tensor(
        [codificar(prompt)],
        dtype=torch.long,
        device=dispositivo
    )
    salida = modelo.generar(
        contexto,
        max_tokens_nuevos=200,
        temperatura=CONFIG["temperatura"],
        top_k=CONFIG["top_k"]
    )
    print(f"\nModelo:\n{decodificar(salida[0].tolist())}\n")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')